In [ ]:
from pathlib import Path

import gstools as gs
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from context_flux_no.simulations.pde import CubicFlux1D
from context_flux_no.simulations.utils import (
    generate_dataset,
)
from context_flux_no.waveforms.grf import (
    GaussianRandomField,
    GaussianRandomField1D,
    PeriodicCov,
)
from context_flux_no.waveforms.step import PeriodicRandomStepFunction1D


jax.config.update("jax_default_device", jax.devices("gpu")[1])

In [ ]:
x = jnp.linspace(0, 1, 100)
grf = GaussianRandomField(
    covariance_fns=[
        gs.Gaussian(dim=1, var=1.0, len_scale=0.2),
    ],
)
u0 = grf.sample(x, jax.random.key(0))

step = PeriodicRandomStepFunction1D()
u0_step = step.sample(x, jax.random.key(2))
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].plot(x, u0[0])
axes[1].plot(x, u0_step[0])

In [ ]:
dataset = generate_dataset(
    dataset_name="cubic_test_compressed",
    n_coeffs=10,
    n_ics_per_coeff=10,
    pde_factory=CubicFlux1D,
    initial_condition_fn=GaussianRandomField(
        covariance_fns=[
            gs.Gaussian(dim=1, var=1.0, len_scale=0.2),
        ],
    ).sample,
    coeff_range_dict={"a": (-1.0, 1.0), "b": (-1.0, 1.0), "c": (-1.0, 1.0)},
    x_spans=[
        (0, 1),
    ],
    Nxs=[
        100,
    ],
    t_span=(0, 0.5),
    Nt=100,
    seed=0,
)

In [ ]:
print(dataset.root.tree())

In [ ]:
print(dataset.root.tree())

In [ ]:
await dataset.root.store.getsize_prefix("t0_fields")

In [ ]:
await dataset.root.store.getsize_prefix("")

In [ ]:
sol = dataset[10]
u = sol["t0_fields"]["u"]

n_plots = 5
inds = np.linspace(0, u.shape[0] - 1, n_plots, dtype=np.int_)
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharex=True)

for j, idx in enumerate(inds):
    axes[j].plot(u[idx])

In [ ]:
dataset["t0_fields"]

In [ ]:
savedir = Path("../../data/")
savedir.mkdir(parents=True, exist_ok=True)
dataset.to_netcdf(savedir / "cubic_no_source_large_train.hdf5")

In [ ]:
dataset_test = generate_dataset(
    n_coeffs=100,
    n_ics_per_coeff=10,
    pde_factory=CubicFlux1D,
    initial_condition_fn=GaussianRandomField1D(PeriodicCov(1.0)).sample,
    coeff_range_dict={"a": (-1.0, 1.0), "b": (-1.0, 1.0), "c": (-1.0, 1.0)},
    x_span=(0, 1),
    Nx=100,
    t_span=(0, 0.5),
    Nt=1000,
    seed=1,
)

savedir = Path("../../data/")
savedir.mkdir(parents=True, exist_ok=True)
dataset_test.to_netcdf(savedir / "cubic_no_source_test.hdf5")

## Concatenate datasets into single file

In [ ]:
datadir = Path("../../data")
basename = "cubic_no_source_large_train"

In [ ]:
ds = xr.open_mfdataset(
    sorted(list(datadir.glob(f"{basename}_seed=*.hdf5"))),
    combine="nested",
    concat_dim="pde",
    engine="h5netcdf",
)

In [ ]:
ds = ds.astype("float32", casting="same_kind")

In [ ]:
ds

In [ ]:
ds.to_netcdf(datadir / f"{basename}.hdf5", engine="h5netcdf")